In [1]:
import glob
import pandas as pd
import os
from pathlib import Path
import glob
import shutil
import re
import unicodedata

In [2]:
def normalize(text):
    if pd.isnull(text):
        return ""
    # Lowercase
    text = text.lower()
    # Remove accents (e.g., é → e)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    # Strip leading/trailing spaces
    return text.strip().replace(' ', '_')

In [3]:
ROOT = '/Users/tomasandrade/Documents/BSC/ICHOIR/datasets/GTSinger'

In [4]:
lang = 'FR'

In [5]:
output_folder = f'{ROOT}/GTSinger_{lang}_flat'

In [28]:
os.makedirs(f'{output_folder}/wav')
os.makedirs(f'{output_folder}/TextGrid')

In [7]:
all_files = glob.glob(f'{ROOT}/{lang}/*/**/***/****/*****')
data = [file.split(f'{ROOT}/{lang}/')[1].split('/') for file in all_files]

df = pd.DataFrame(data = data, columns=['singer', 'style', 'song', 'group', 'file'])
df['input_path'] = all_files


# prepare output paths
df['song'] = df['song'].apply(normalize)
df['new_song'] = df['style'] + '_' +  df['song'] + '_' + df['group']
df['out_path'] = df['singer'] + '-' + df['new_song'] + '-' + df['file'].str.replace('_TextGrid', '.TextGrid')

In [8]:
df['style'].unique()

array(['Vibrato', 'Pharyngeal', 'Breathy', 'Glissando',
       'Mixed_Voice_and_Falsetto'], dtype=object)

In [24]:
SELECT_STYLES = ['Vibrato', 'Glissando']
EXCLUDE_GROUPS = ['Paired_Speech_Group', 'Control_Group']

df_songs = df[~df['group'].isin(EXCLUDE_GROUPS)]

df_songs = df_songs[df_songs['style'].isin(SELECT_STYLES)]

df_wav = df_songs[df_songs['file'].str.contains('wav')].sort_values('input_path')
df_wav['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/wav/' + df_wav['out_path']

df_TextGrid = df_songs[df_songs['file'].str.contains('TextGrid')].sort_values('input_path')
df_TextGrid['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/TextGrid/' + df_TextGrid['out_path']

In [25]:
df_wav

,singer,style,song,group,file,input_path,new_song,out_path
3060,FR-Soprano-1,Glissando,le_voyage_commence,Glissando_Group,0000.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_le_voyage_commence_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
3059,FR-Soprano-1,Glissando,le_voyage_commence,Glissando_Group,0001.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_le_voyage_commence_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
3051,FR-Soprano-1,Glissando,le_voyage_commence,Glissando_Group,0002.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_le_voyage_commence_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
3053,FR-Soprano-1,Glissando,le_voyage_commence,Glissando_Group,0003.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_le_voyage_commence_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
3041,FR-Soprano-1,Glissando,le_voyage_commence,Glissando_Group,0004.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Glissando_le_voyage_commence_Glissando_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
...,...,...,...,...,...,...,...,...
6286,FR-Tenor-1,Vibrato,quel_trouble,Vibrato_Group,0016.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_quel_trouble_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
6288,FR-Tenor-1,Vibrato,quel_trouble,Vibrato_Group,0017.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_quel_trouble_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
6338,FR-Tenor-1,Vibrato,quel_trouble,Vibrato_Group,0018.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_quel_trouble_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
6335,FR-Tenor-1,Vibrato,quel_trouble,Vibrato_Group,0019.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_quel_trouble_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...


In [29]:
df_wav.shape, df_TextGrid.shape

((394, 8), (394, 8))

In [30]:
for input_path, out_path in zip(df_wav['input_path'], df_wav['out_path']):
    shutil.copy2(input_path, out_path)

for input_path, out_path in zip(df_TextGrid['input_path'], df_TextGrid['out_path']):
    shutil.copy2(input_path, out_path)

In [31]:
lang

'FR'

# Checks

In [32]:
def check_matching_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    wav_root = f'{ROOT}/GTSinger_{lang}_flat/wav'

    tg_files = sorted(glob.glob(f'{tg_root}/*'))
    wav_files = sorted(glob.glob(f'{wav_root}/*'))

    tg_stems = [file.split('/')[-1].split('.TextGrid')[0] for file in tg_files]
    wav_stems = [file.split('/')[-1].split('.wav')[0] for file in wav_files]

    df_tg = pd.DataFrame(data = tg_stems)
    df_wav = pd.DataFrame(data = wav_stems)

    n_wav = len(df_wav)
    n_tg = len(df_tg)

    matches = (df_tg == df_wav).sum().values[0]

    return n_wav, n_tg, matches

In [33]:
check_matching_files('FR')

(394, 394, 394)

In [34]:
from textgrids import TextGrid

In [35]:
def get_all_tg_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    tg_files = sorted(glob.glob(f'{tg_root}/*'))

    return tg_files

def get_total_length(lang):
    files = get_all_tg_files(lang)
    secs = sum([get_tg_length(f) for f in files])
    return secs/60.

def get_tg_length(file):
    tg = TextGrid()
    tg.read(file)

    return tg['global'].xmax

In [36]:
get_total_length('FR')

80.59560000000002